<table style="width:100%">
<tr>                                                                                   
     <th>
        <div style="padding:15px;color:#030aa7;font-size:240%;text-align:center;font-style:italic;font-weight:bold;font-family:Georgia,serif">GEOLOCATION</div>
     </th>
 </tr>
<tr>                                                                                   
     <th><img src="https://raw.githubusercontent.com/rbizoi/PythonFormation/main/images/e-brasil.png" width="850"></th>
 </tr>    
</table>

# Les imports et initialisations des variables

In [1]:
from datetime import datetime
import pandas as pd, numpy as np, os, warnings, seaborn as sns, pickle, re, unicodedata
from datetime import datetime as dt
import sqlalchemy

warnings.filterwarnings(action="ignore")

## Changement de répertoire

In [2]:
os.chdir("/opt/spark/data/ebrasil")

# DataFrame $geolocation$

In [3]:
donnees = pd.read_csv('olist_geolocation_dataset.csv')

In [4]:
donnees.shape

(720478, 5)

In [5]:
donnees.zip_code.nunique()

19015

In [6]:
donnees[['zip_code','city','state']].drop_duplicates().shape

(19581, 3)

In [7]:
geographie = donnees.groupby('zip_code').agg({
    'lat': ['min', 'median', 'max'],
    'lng': ['min', 'median', 'max'],
    'city': 'first',
    'state': 'first'
}).reset_index()
geographie.columns = [geographie.columns[0][0]]+[ col[0]+'_'+col[1] for col in geographie.columns[1:] ]
geographie.columns = [col.replace('_first','').replace('_median','') for col in geographie.columns]
geographie = geographie[['zip_code','city','state','lat','lng','lat_min','lat_max','lng_min','lng_max']]
geographie.head()

,zip_code,city,state,lat,lng,lat_min,lat_max,lng_min,lng_max
0,1001,sao paulo,São Paulo,-23.550107,-46.634027,-23.551427,-23.549292,-46.634410,-46.633559
1,1002,sao paulo,São Paulo,-23.548228,-46.635247,-23.548878,-23.544641,-46.636361,-46.633180
2,1003,sao paulo,São Paulo,-23.548976,-46.635318,-23.549083,-23.548901,-46.637157,-46.634862
3,1004,sao paulo,São Paulo,-23.549550,-46.634771,-23.550765,-23.549181,-46.635371,-46.634057
4,1005,sao paulo,São Paulo,-23.549690,-46.636532,-23.549980,-23.548758,-46.638411,-46.634768


In [8]:
geographie.zip_code.nunique()

19015

In [9]:
donnees = geographie

## Sauvegarde en parquet

In [10]:
os.chdir("/opt/spark/notebooks/ecommerce")

In [11]:
donnees.to_parquet('geolocation.parquet',compression='gzip', engine='pyarrow')

In [12]:
pd.read_parquet('geolocation.parquet').info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19015 entries, 0 to 19014
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   zip_code  19015 non-null  int64  
 1   city      19015 non-null  object 
 2   state     19015 non-null  object 
 3   lat       19015 non-null  float64
 4   lng       19015 non-null  float64
 5   lat_min   19015 non-null  float64
 6   lat_max   19015 non-null  float64
 7   lng_min   19015 non-null  float64
 8   lng_max   19015 non-null  float64
dtypes: float64(6), int64(1), object(2)
memory usage: 1.3+ MB


In [13]:
!ls -al

total 35560
drwxr-xr-x 1 spark spark     512 Sep 26 16:53 .
drwxrwxrwx 1 root  root      512 Sep 26 16:50 ..
-rw-r--r-- 1 spark spark 4508308 Sep 26 16:51 customers.parquet
-rw-r--r-- 1 spark spark 1120853 Sep 26 16:53 geolocation.parquet
-rw-r--r-- 1 spark spark 4773153 Sep 26 16:52 items.parquet
-rw-r--r-- 1 spark spark 9489433 Sep 26 16:51 orders.parquet
-rw-r--r-- 1 spark spark 2466806 Sep 26 16:52 paymentsI.parquet
-rw-r--r-- 1 spark spark 2648930 Sep 26 16:52 payments.parquet
-rw-r--r-- 1 spark spark  958461 Sep 26 16:52 products.parquet
-rw-r--r-- 1 spark spark 7490087 Sep 26 16:52 reviews.parquet
-rw-r--r-- 1 spark spark 2847395 Sep 26 16:52 reviews_p.parquet
-rw-r--r-- 1 spark spark   91207 Sep 26 16:52 sellers.parquet
